NLA round-trip accuracy degradation experiment.

Idea: patch a single token's activation at a single layer with its
NLA round-trip reconstruction 
$$
(h -> AV(h) -> text -> AR(text) -> h_hat)
$$
then let generation continue normally. Compare downstream task accuracy
against (a) an unpatched baseline and (b) a no-op patch control that
re-injects the *original* activation unchanged.

This script assumes you have:
  - a HF-loadable Gemma model (`MODEL_NAME`)
  - your own trained AV and AR modules from natural_language_autoencoders,
    exposed as callables: av(h_vec) -> str, ar(text) -> torch.Tensor
  - GSM8K test split via `datasets`

Fill in the two TODOs (load_av_ar, extract_final_answer) for your repo's
actual API / answer-parsing format before running.

## Experiment Overview

All experiments share one AV server and use the same target tokenizer/model state where possible. The notebook deliberately unloads the 27B target before loading the AR, then reloads the target for introspection, so the full target and AR do not coexist on the notebook GPU.

Results are printed and displayed near the cells that produce them. Round-trip and paraphrasing summaries are also written beneath the configurable experiment output root; introspection and zero-vector results remain notebook-only.

In [1]:
from __future__ import annotations

# -----------------------------------------------------------------------------
# EDIT THESE VALUES FOR YOUR CLUSTER ALLOCATION
# -----------------------------------------------------------------------------
# USER_NAME is your Linux username / /home folder name.
# VISIBLE_GPUS are the physical GPU IDs assigned to you, comma-separated.
# After CUDA masking, the first listed physical GPU becomes cuda:0 in this notebook.
# SGLANG_PHYSICAL_GPU is the physical GPU reserved for the SGLang AV server.
USER_NAME = "kaylee"
VISIBLE_GPUS = "0,1"
SGLANG_PHYSICAL_GPU = "1"
LOCAL_DEVICE = "cuda:0"
# -----------------------------------------------------------------------------

# Run this notebook from a fresh kernel. CUDA_VISIBLE_DEVICES must be set before
# importing torch, so this setup cell intentionally defines paths/GPUs first.
import os
from pathlib import Path

HOME = Path("/home") / USER_NAME
NLA_REPO = Path(os.environ.get("NLA_REPO", HOME / "natural_language_autoencoders"))
SGLANG_REPO = Path(os.environ.get("SGLANG_REPO", HOME / "sglang"))
VENV_PYTHON = Path(os.environ.get("NLA_VENV_PYTHON", NLA_REPO / ".venv/bin/python"))
HF_CACHE = Path(os.environ.get("HF_HOME", HOME / ".cache/huggingface"))
ROOT = Path(os.environ.get("NLA_EXPERIMENT_ROOT", HOME / "experiments/nla_experiments"))
LIBNUMA_DIR = Path(
    os.environ.get(
        "NLA_LIBNUMA_DIR", NLA_REPO / "vendor/libnuma/usr/lib/x86_64-linux-gnu"
    )
)
PYTHON_INCLUDE_DIRS = [
    Path(
        os.environ.get(
            "NLA_PYTHON_INCLUDE",
            NLA_REPO / "vendor/python3.10-dev/usr/include/python3.10",
        )
    ),
    Path(
        os.environ.get(
            "NLA_PYTHON_MULTIARCH_INCLUDE",
            NLA_REPO / "vendor/python3.10-dev/usr/include",
        )
    ),
]
ROOT.mkdir(parents=True, exist_ok=True)


os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = VISIBLE_GPUS
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
if LIBNUMA_DIR.exists():
    os.environ["LD_LIBRARY_PATH"] = (
        f"{LIBNUMA_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"
    )
_existing_cpath = os.environ.get("CPATH", "")
_include_paths = [str(path) for path in PYTHON_INCLUDE_DIRS if path.exists()]
if _include_paths:
    os.environ["CPATH"] = ":".join([*_include_paths, _existing_cpath])

import gc
import math
import random
import re
import subprocess
import time
from contextlib import contextmanager
from typing import Any

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# Local NLA inference helpers from the released repo.
import sys

sys.path.insert(0, str(NLA_REPO))
from nla_inference import NLAClient, NLACritic

TARGET_MODEL = "google/gemma-3-27b-it"
ACTOR_REPO_ID = os.environ.get("NLA_ACTOR_DIR", "kitft/nla-gemma3-27b-L41-av")
AR_REPO_ID = os.environ.get("NLA_AR_DIR", "kitft/nla-gemma3-27b-L41-ar")
SGLANG_URL = os.environ.get("SGLANG_URL", "http://127.0.0.1:30000")
SGLANG_LOG = Path.home() / "logs" / "sglang_av_server.log"
SGLANG_LOG.parent.mkdir(parents=True, exist_ok=True)
LAYER_INDEX = 41
DEVICE = LOCAL_DEVICE
SEED = 1234
N_EXAMPLES = 200


def resolve_checkpoint(path_or_repo_id: str) -> str:
    """Return a local checkpoint path with nla_meta.yaml available."""
    candidate = Path(path_or_repo_id).expanduser()
    if candidate.exists():
        assert (
            candidate / "nla_meta.yaml"
        ).exists(), f"Missing nla_meta.yaml in {candidate}"
        return str(candidate)
    local_path = Path(snapshot_download(repo_id=path_or_repo_id))
    assert (
        local_path / "nla_meta.yaml"
    ).exists(), f"Missing nla_meta.yaml in downloaded snapshot {local_path}"
    return str(local_path)


ACTOR_DIR = resolve_checkpoint(ACTOR_REPO_ID)
AR_DIR = resolve_checkpoint(AR_REPO_ID)

assert NLA_REPO.exists(), f"Missing NLA repo: {NLA_REPO}"
assert SGLANG_REPO.exists(), f"Missing patched SGLang checkout: {SGLANG_REPO}"
assert VENV_PYTHON.exists(), f"Missing NLA venv Python: {VENV_PYTHON}"

print("home:", HOME)
print("NLA repo:", NLA_REPO)
print("SGLang checkout:", SGLANG_REPO)
print("HF cache:", HF_CACHE)
print("experiment output:", ROOT)
print("vendored libnuma:", LIBNUMA_DIR if LIBNUMA_DIR.exists() else "not found")
print("vendored Python includes:", [p for p in PYTHON_INCLUDE_DIRS if p.exists()])
print("visible physical GPUs:", VISIBLE_GPUS)
print("SGLang physical GPU:", SGLANG_PHYSICAL_GPU)
print("notebook device:", DEVICE)
print("actor checkpoint:", ACTOR_DIR)
print("AR checkpoint:", AR_DIR)
print(torch.__version__, torch.cuda.is_available())
if torch.cuda.is_available():
    print("visible cuda devices:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i}", torch.cuda.get_device_name(i))

UV_PYTHON_INCLUDE = "/home/kaylee/.local/share/uv/python/cpython-3.10.20-linux-x86_64-gnu/include/python3.10"
assert os.path.exists(os.path.join(UV_PYTHON_INCLUDE, "Python.h")), "Python.h not found at that path"

os.environ["CPATH"] = f"{UV_PYTHON_INCLUDE}:{os.environ.get('CPATH', '')}"
print("CPATH now:", os.environ["CPATH"])

REAL_LIBNUMA_DIR = "/home/kaylee/.local/lib"
assert os.path.exists(os.path.join(REAL_LIBNUMA_DIR, "libnuma.so.1")), "libnuma.so.1 not found there"

os.environ["LD_LIBRARY_PATH"] = f"{REAL_LIBNUMA_DIR}:{os.environ.get('LD_LIBRARY_PATH', '')}"
print("LD_LIBRARY_PATH now:", os.environ["LD_LIBRARY_PATH"])

!rm -rf ~/.triton/cache

print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH"))
print("CPATH:", os.environ.get("CPATH"))

Fetching 38 files:   0%|          | 0/38 [00:00<?, ?it/s]

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

home: /home/kaylee
NLA repo: /home/kaylee/natural_language_autoencoders
SGLang checkout: /home/kaylee/sglang
HF cache: /home/kaylee/.cache/huggingface
experiment output: /home/kaylee/experiments/nla_experiments
vendored libnuma: not found
vendored Python includes: []
visible physical GPUs: 0,1
SGLang physical GPU: 1
notebook device: cuda:0
actor checkpoint: /home/kaylee/.cache/huggingface/hub/models--kitft--nla-gemma3-27b-L41-av/snapshots/4e721238131ffb8348cff260fe81b8b34a270a0d
AR checkpoint: /home/kaylee/.cache/huggingface/hub/models--kitft--nla-gemma3-27b-L41-ar/snapshots/aa2f29723c4807caf5f665dac004998df71b7cfb
2.9.1+cu128 True
visible cuda devices: 2
cuda:0 NVIDIA A100-SXM4-80GB
cuda:1 NVIDIA A100-SXM4-80GB
CPATH now: /home/kaylee/.local/share/uv/python/cpython-3.10.20-linux-x86_64-gnu/include/python3.10:
LD_LIBRARY_PATH now: /home/kaylee/.local/lib:
LD_LIBRARY_PATH: /home/kaylee/.local/lib:
CPATH: /home/kaylee/.local/share/uv/python/cpython-3.10.20-linux-x86_64-gnu/include/python

## Start The AV Server

The AV uses SGLang because the checkpoint expects `input_embeds` injection. This notebook launches the patched editable SGLang checkout through the NLA `.venv`, pinned to the physical GPU chosen in the setup cell.



In [2]:
def launch_sglang_actor() -> subprocess.Popen:
    env = os.environ.copy()
    env["HF_HOME"] = str(HF_CACHE)
    env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    env["CUDA_VISIBLE_DEVICES"] = SGLANG_PHYSICAL_GPU
    env["SGLANG_MIN_NEW_TOKEN_RATIO_FACTOR"] = "1"
    env["PYTHONPATH"] = f"{NLA_REPO}:{env.get('PYTHONPATH', '')}"
    if LIBNUMA_DIR.exists():
        env["LD_LIBRARY_PATH"] = f"{LIBNUMA_DIR}:{env.get('LD_LIBRARY_PATH', '')}"
    include_paths = [str(path) for path in PYTHON_INCLUDE_DIRS if path.exists()]
    if include_paths:
        env["CPATH"] = ":".join([*include_paths, env.get("CPATH", "")])

    cmd = [
        str(VENV_PYTHON),
        "-m",
        "sglang.launch_server",
        "--model-path",
        ACTOR_DIR,
        "--port",
        "30000",
        "--host",
        "127.0.0.1",
        "--disable-radix-cache",
        "--mem-fraction-static",
        "0.80",
        "--context-length",
        "512",
        "--attention-backend",
        "triton",
        "--disable-cuda-graph",
        "--trust-remote-code",
        "--log-level",
        "warning",
    ]
    print("Launching SGLang on physical GPU", SGLANG_PHYSICAL_GPU)
    print(" ".join(cmd))
    print("SGLang server logs ->", SGLANG_LOG)
    log_f = open(SGLANG_LOG, "ab", buffering=0)
    return subprocess.Popen(
        cmd, env=env, cwd=str(NLA_REPO), stdout=log_f, stderr=subprocess.STDOUT
    )


def sglang_is_healthy() -> bool:
    import urllib.request

    try:
        urllib.request.urlopen(SGLANG_URL + "/health", timeout=2).read()
        return True
    except Exception:
        return False


def wait_for_sglang(timeout_s: int = 300) -> None:
    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        if sglang_is_healthy():
            print("SGLang is healthy")
            return
        try:
            import urllib.request

            urllib.request.urlopen(SGLANG_URL + "/health", timeout=2).read()
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise TimeoutError(f"SGLang did not become healthy: {last_error!r}")


sglang_proc = None
if sglang_is_healthy():
    print("SGLang is already healthy at", SGLANG_URL)
else:
    sglang_proc = launch_sglang_actor()
    wait_for_sglang()

Launching SGLang on physical GPU 1
/home/kaylee/natural_language_autoencoders/.venv/bin/python -m sglang.launch_server --model-path /home/kaylee/.cache/huggingface/hub/models--kitft--nla-gemma3-27b-L41-av/snapshots/4e721238131ffb8348cff260fe81b8b34a270a0d --port 30000 --host 127.0.0.1 --disable-radix-cache --mem-fraction-static 0.80 --context-length 512 --attention-backend triton --disable-cuda-graph --trust-remote-code --log-level warning
SGLang server logs -> /home/kaylee/logs/sglang_av_server.log


SGLang is healthy


In [3]:
print(sglang_is_healthy())

True


# Experiment 1

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.bfloat16, device_map={"": DEVICE},
    trust_remote_code=True, attn_implementation="eager",
).eval()
del model.model.vision_tower
gc.collect()
torch.cuda.empty_cache()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

model-00001-of-00012.safetensors:   0%|          | 0.00/4.85G [00:00<?, ?B/s]

model-00004-of-00012.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

...

model-00010-of-00012.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00011-of-00012.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00012-of-00012.safetensors:   0%|          | 0.00/462M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [5]:
print("model" in globals())
if "model" in globals():
    print(model.device if hasattr(model, "device") else next(model.parameters()).device)

True
cuda:0


In [6]:
client = NLAClient(ACTOR_DIR, sglang_url=SGLANG_URL)
critic = NLACritic(AR_DIR, device="cpu", dtype=torch.float32)
print("Target and AV client ready")


[NLAClient] 4e721238131ffb8348cff260fe81b8b34a270a0d: d_model=5376 inj_scale=60000.0 embed_scale=73.32 inj_char='㈜'(id=246566)


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Some weights of Gemma3ForCausalLM were not initialized from the model checkpoint at /home/kaylee/.cache/huggingface/hub/models--kitft--nla-gemma3-27b-L41-ar/snapshots/aa2f29723c4807caf5f665dac004998df71b7cfb and are newly initialized: ['model.norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[NLACritic] 42 layers  d_model=5376  mse_scale=73.32
Target and AV client ready


The reconstruction error is
$$
\mathcal{L} = \mathbb{E}_{h_l \sim \mathcal{H}} \mathbb{E}_{z \sim AV(\cdot | h_l)} [||h_l - AR(z)||_2^2]
$$
where $\mathcal{H}$ is the distribution produced by extracting layer $l$ activations from $M$ on a corpus of text. 

The FVE is
$$
1 - \frac{L}{ \mathbb{E}_{h_l \sim \mathcal{H}} ||h_l - \bar{h_l}||^2_2}
$$

In [7]:
N_SANITY = 200
MAX_NEW_TOKENS = 256

def av(h: torch.Tensor) -> str:
    v = h.detach().float().cpu().numpy()
    return client.generate(v, temperature=0.0, max_new_tokens=MAX_NEW_TOKENS, extract_explanation=True)

PARAPHRASE_PROMPT = (
    "Paraphrase the following explanation, preserving its meaning exactly "
    "but using different wording:\n\n{text}\n\nParaphrase:"
)

def paraphrase(text: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    prompt = PARAPHRASE_PROMPT.format(text=text)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def ar(text: str) -> torch.Tensor:
    return critic.reconstruct(text)  # raw, unnormalized -- matches h's scale


# --- Patch hook: replace the residual stream at the last prompt token ---
class LastTokenPatcher:
    def __init__(self, patch_vec=None):
        self.patch_vec = patch_vec
        self.captured = None
        self._done = False  # only patch once

    def __call__(self, module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        # Only act on the initial multi-token prefill pass, not each
        # single-token decode step that follows.
        if hidden.shape[1] > 1 and not self._done:
            self.captured = hidden[0, -1, :].detach().clone()
            if self.patch_vec is not None:
                hidden = hidden.clone()
                hidden[0, -1, :] = self.patch_vec.to(hidden.dtype).to(hidden.device)
            self._done = True
            return (hidden,) + output[1:] if isinstance(output, tuple) else hidden
        return output


def mse_cos_from_reconstruction(h_hat: torch.Tensor, h_orig: torch.Tensor, mse_scale: float):
    """Per-example, norm-matched metrics (for reporting alongside FVE, not used *in* FVE)."""
    pred = h_hat.float()
    gold = h_orig.float().cpu()
    pred_n = pred / pred.norm().clamp_min(1e-12) * mse_scale
    gold_n = gold / gold.norm().clamp_min(1e-12) * mse_scale
    mse = ((pred_n - gold_n) ** 2).mean().item()
    cos = (pred_n @ gold_n / (pred_n.norm() * gold_n.norm())).item()
    return mse, cos


def raw_sq_error(h_hat: torch.Tensor, h_orig: torch.Tensor) -> float:
    """||h_orig - h_hat||^2 on raw, unnormalized activations -- this is what
    the paper's L and FVE are defined over, so it must NOT be norm-rescaled
    the way mse_cos_from_reconstruction's inputs are."""
    assert h_hat.dim() == 1 and h_orig.dim() == 1, f"{h_hat.shape=} {h_orig.shape=}"
    return ((h_orig.float().cpu() - h_hat.float().cpu()) ** 2).sum().item()


def fve(sq_errors: list, h_orig_list: list) -> float:
    """FVE = 1 - E[||h - AR(z)||^2] / E[||h - h_bar||^2], both expectations
    taken over the same sample. h_bar is estimated as the sample mean of
    h_orig_list -- this must be computed over the full run, not per-example."""
    assert h_hat.dim() == 1 and h_orig.dim() == 1, f"{h_hat.shape=} {h_orig.shape=}"
    h_stack = torch.stack([h.float().cpu() for h in h_orig_list])  # [N, d]
    h_bar = h_stack.mean(dim=0)
    variance = ((h_stack - h_bar) ** 2).sum(dim=1).mean().item()
    L = sum(sq_errors) / len(sq_errors)
    return 1 - L / variance


def get_layer_module(m, layer_idx):
    return m.model.language_model.layers[layer_idx]

# https://github.com/kitft/natural_language_autoencoders/blob/main/examples/gemma27b_layer41_step6000.txt
VAR_NRM = 0.0579  # Gemma-3-27B / layer-41 predict-the-mean MSE from training set

def fve_nrm(mse_nrm: float) -> float:
    """Fraction of training-set activation variance explained (normalized space)."""
    return 1.0 - float(mse_nrm) / VAR_NRM


def run_with_patch(prompt: str, patch_vec, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    patcher = LastTokenPatcher(patch_vec)
    handle = get_layer_module(model, LAYER_INDEX).register_forward_hook(patcher)
    try:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    finally:
        handle.remove()
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, patcher.captured


# --- Run the three-condition comparison on a GSM8K subset ---
import re
from datasets import load_dataset

def extract_final_answer(text: str):
    m = re.search(r"[-+]?\d[\d,]*\.?\d*", text.strip().split("\n")[-1])
    return m.group().replace(",", "") if m else None

def gsm8k_gold(ex):
    return ex["answer"].split("####")[-1].strip().replace(",", "")

ds = load_dataset("gsm8k", "main", split="test").select(range(N_SANITY))
correct = {"baseline": 0, "nla_roundtrip": 0, "paraphrase_roundtrip": 0}

h_orig_list = []
h_hat_list, h_hat_para_list = [], []
cos_list, mse_list, sq_err_list = [], [], []
cos_para_list, mse_para_list, sq_err_para_list = [], [], []
mse_nrm_list, cos_nrm_list = [], []              # new
mse_nrm_para_list, cos_nrm_para_list = [], []    # new
nla_correct_list, paraphrase_correct_list = [], []

from tqdm.auto import tqdm

for i, ex in enumerate(tqdm(ds, desc="Evaluating", unit="ex")):
    prompt = ex["question"] + "\nAnswer: Let's think step by step."
    gold = gsm8k_gold(ex)

    text_base, h_orig = run_with_patch(prompt, patch_vec=None, max_new_tokens=MAX_NEW_TOKENS)
    correct["baseline"] += extract_final_answer(text_base) == gold
    h_orig_list.append(h_orig)

    explanation = av(h_orig)

    # Condition 2: plain round-trip
    h_hat = ar(explanation)
    h_hat_list.append(h_hat)
    mse, cos = mse_cos_from_reconstruction(h_hat, h_orig, critic.mse_scale)
    mse_list.append(mse); cos_list.append(cos)
    sq_err_list.append(raw_sq_error(h_hat, h_orig))

    mse_nrm, cos_nrm = critic.score(explanation, h_orig.detach().float().cpu())
    mse_nrm_list.append(mse_nrm); cos_nrm_list.append(cos_nrm)

    text_nla, _ = run_with_patch(prompt, patch_vec=h_hat, max_new_tokens=MAX_NEW_TOKENS)
    is_correct = extract_final_answer(text_nla) == gold
    nla_correct_list.append(is_correct)
    correct["nla_roundtrip"] += is_correct

    # Condition 3: paraphrase round-trip
    paraphrased = paraphrase(explanation)
    h_hat_para = ar(paraphrased)
    h_hat_para_list.append(h_hat_para)
    mse_p, cos_p = mse_cos_from_reconstruction(h_hat_para, h_orig, critic.mse_scale)
    mse_para_list.append(mse_p); cos_para_list.append(cos_p)
    sq_err_para_list.append(raw_sq_error(h_hat_para, h_orig))

    mse_nrm_p, cos_nrm_p = critic.score(paraphrased, h_orig.detach().float().cpu())
    mse_nrm_para_list.append(mse_nrm_p); cos_nrm_para_list.append(cos_nrm_p)

    text_para, _ = run_with_patch(prompt, patch_vec=h_hat_para, max_new_tokens=MAX_NEW_TOKENS)
    is_correct_para = extract_final_answer(text_para) == gold
    paraphrase_correct_list.append(is_correct_para)
    correct["paraphrase_roundtrip"] += is_correct_para

n = N_SANITY

fve_roundtrip = fve(sq_err_list, h_orig_list)
fve_paraphrase = fve(sq_err_para_list, h_orig_list)

fve_nrm_roundtrip = fve_nrm(sum(mse_nrm_list) / len(mse_nrm_list))            # new
fve_nrm_paraphrase = fve_nrm(sum(mse_nrm_para_list) / len(mse_nrm_para_list)) # new

print(f"{'Condition':<22}{'Accuracy':<12}{'Cos-sim':<12}{'MSE':<12}{'FVE':<10}{'FVE (nrm)':<12}")
print(f"{'Baseline':<22}{correct['baseline']/n:<12.2%}{'--':<12}{'--':<12}{'--':<10}{'--':<12}")
print(f"{'NLA round-trip':<22}{correct['nla_roundtrip']/n:<12.2%}"
      f"{sum(cos_list)/len(cos_list):<12.3f}{sum(mse_list)/len(mse_list):<12.4f}"
      f"{fve_roundtrip:<10.3f}{fve_nrm_roundtrip:<12.3f}")
print(f"{'Paraphrase round-trip':<22}{correct['paraphrase_roundtrip']/n:<12.2%}"
      f"{sum(cos_para_list)/len(cos_para_list):<12.3f}{sum(mse_para_list)/len(mse_para_list):<12.4f}"
      f"{fve_paraphrase:<10.3f}{fve_nrm_paraphrase:<12.3f}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Evaluating:   0%|          | 0/200 [00:00<?, ?ex/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

Standard reconstruction metrics disagree sharply depending on whether they're scale-normalized: cosine similarity (0.992) indicates the AR recovers the activation's direction well, while raw-scale FVE is strongly negative because 81% of total squared error is concentrated in just 10 of 5,376 dimensions — a small set of "massive activation" dimensions with near-constant, very large magnitude across examples. This suggests the AR is not well-calibrated on these specific outlier dimensions, even though it captures the semantically relevant structure of the activation.

In [ ]:
labels = ["Baseline", "NLA round-trip", "Paraphrase round-trip"]
values = [correct["baseline"]/n, correct["nla_roundtrip"]/n, correct["paraphrase_roundtrip"]/n]

plt.figure(figsize=(5,4))
plt.bar(labels, values, color=["#888", "#7F77DD", "#DD9E4F"])
plt.ylabel("Accuracy")
plt.ylim(0, 1)
for i, v in enumerate(values):
    plt.text(i, v - 0.08, f"{v:.0%}", ha="center")
plt.title(f"GSM8K accuracy (n={n})")
plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()

# --- Reconstruction-quality summary: cos-sim, MSE, FVE per condition ---
# Baseline has no reconstruction, so it's excluded from these three.
recon_labels = ["NLA round-trip", "Paraphrase round-trip"]
recon_colors = ["#7F77DD", "#DD9E4F"]

cos_means = [sum(cos_list)/len(cos_list), sum(cos_para_list)/len(cos_para_list)]
mse_means = [sum(mse_list)/len(mse_list), sum(mse_para_list)/len(mse_para_list)]
fve_vals = [fve_roundtrip, fve_paraphrase]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].bar(recon_labels, cos_means, color=recon_colors)
axes[0].set_ylabel("Cosine similarity")
axes[0].set_ylim(0, 1)
axes[0].set_title("Mean cos-sim")
for i, v in enumerate(cos_means):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center")

axes[1].bar(recon_labels, mse_means, color=recon_colors)
axes[1].set_ylabel("MSE")
axes[1].set_title("Mean MSE")
for i, v in enumerate(mse_means):
    axes[1].text(i, v, f"{v:.4f}", ha="center", va="bottom")

axes[2].bar(recon_labels, fve_vals, color=recon_colors)
axes[2].set_ylabel("FVE")
axes[2].set_ylim(0, 1)
axes[2].set_title("Fraction of variance explained")
for i, v in enumerate(fve_vals):
    axes[2].text(i, v + 0.02, f"{v:.3f}", ha="center")

for ax in axes:
    ax.tick_params(axis='x', rotation=15)

plt.suptitle(f"Reconstruction quality by condition (n={n})")
plt.tight_layout()
plt.savefig("reconstruction_quality_summary.png", dpi=150)
plt.show()

# --- Per-example cos-sim vs correctness, one panel per reconstruction condition ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

panel_data = [
    (cos_list, nla_correct_list, "NLA round-trip"),
    (cos_para_list, paraphrase_correct_list, "Paraphrase round-trip"),
]

for ax, (cos_vals, correct_flags, title) in zip(axes, panel_data):
    colors = ["#639922" if c else "#E24B4A" for c in correct_flags]
    y_jitter = 1 + np.random.uniform(-0.05, 0.05, size=len(cos_vals))
    ax.scatter(cos_vals, y_jitter, c=colors, s=80, alpha=0.8)
    ax.set_ylim(0.8, 1.2)
    ax.set_yticks([])
    ax.set_xlabel("Cosine similarity (reconstruction vs original)")
    ax.set_title(title)

plt.suptitle("Reconstruction quality per example\n(green=correct, red=incorrect)")
plt.tight_layout()
plt.savefig("cos_vs_correctness.png", dpi=150)
plt.show()

In [ ]:
# Hardcoding due to cut off
cos_means = [0.992, 0.982]
mse_means = [0.0164, 0.0361]
fve_vals = [-10.963, -5.030]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].bar(recon_labels, cos_means, color=recon_colors)
axes[0].set_ylabel("Cosine similarity")
axes[0].set_ylim(0, 1)
axes[0].set_title("Mean cos-sim")
for i, v in enumerate(cos_means):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center")

axes[1].bar(recon_labels, mse_means, color=recon_colors)
axes[1].set_ylabel("MSE")
axes[1].set_title("Mean MSE")
for i, v in enumerate(mse_means):
    axes[1].text(i, v, f"{v:.4f}", ha="center", va="bottom")

axes[2].bar(recon_labels, fve_vals, color=recon_colors)
axes[2].set_ylabel("FVE")
axes[2].set_title("Fraction of variance explained")
for i, v in enumerate(fve_vals):
    axes[2].text(i, v + (0.3 if v >= 0 else -0.6), f"{v:.3f}", ha="center")

for ax in axes:
    ax.tick_params(axis='x', rotation=15)

plt.suptitle(f"Reconstruction quality by condition (n={n})")
plt.tight_layout()
plt.savefig("reconstruction_quality_summary.png", dpi=150)
plt.show()

## New Tasks

We construct a diverse array of over 40 relatively simple tasks to test whether function
vectors can be extracted in diverse settings. To simplify the presentation of our analysis, we focus on
a representative sample of 6 tasks:
- Antonym. Given an input word, generate the word with opposite meaning.
    - Nguyen et al (2017)
- Capitalize. Given an input word, generate the same word with a capital first letter.
    - ChatGPT
- Country-Capital. Given a country name, generate the capital city.
    - ChatGPT
- English-French. Given an English word, generate the French translation of the word.
    - Conneau et al (2017)
- Present-Past. Given a verb in present tense, generate the verb’s simple past inflection.
    - ChatGPT
- Singular-Plural. Given a singular noun, generate its plural inflection
    - ChatGPT


For each task have a plot for number of shots (X axis) and accuracy (y-axis). 

In [ ]:
"""
Shot-count sweep version of the NLA round-trip eval on the 6 ICL
function-vector tasks (Todd et al. 2023, arXiv:2310.15213).

For each task, sweeps the number of ICL demonstrations (x-axis) and
plots accuracy (y-axis) for two conditions:
  - nla_roundtrip:        h patched with AR(verbalize(h))
  - paraphrase_roundtrip: h patched with AR(paraphrase(verbalize(h)))

Produces one PNG per task plus a combined 2x3 grid, and caches raw
results to JSON so you can replot without rerunning the model.

Notebook cells -- paste/import your existing definitions (av,
paraphrase, ar, run_with_patch, mse_cos_from_reconstruction,
raw_sq_error, fve, fve_nrm, LastTokenPatcher, client, critic, model,
tokenizer) before running this.

Dataset sources, all expected flat under datasets/:
  - antonym:           Nguyen et al. 2017 (AntSynNET) -- github.com/nguyenkh/AntSynNET
  - capitalize:        ChatGPT-generated
  - country-capital:   ChatGPT-generated
  - english-french:    Conneau et al. 2017 (MUSE bilingual dictionaries) --
                        both en-fr.0-5000.txt (train) and en-fr.5000-6500.txt
                        (test) splits, concatenated into one pool
  - present-past:      ChatGPT-generated
  - singular-plural:   ChatGPT-generated
"""

# %%
import json
import random
import string
from pathlib import Path

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

TASKS = [
    "antonym",
    "capitalize",
    "country-capital",
    "english-french",
    "present-past",
    "singular-plural",
]

# Shot counts to sweep. 0 = zero-shot (no demonstrations, just "Q: <word>\nA:").
# Start smaller ([0, 1, 4, 16] with N_PER_CONFIG=20) to sanity-check before a full run.
SHOT_COUNTS = [0, 1, 2, 4, 8, 16]
N_PER_CONFIG = 50        # queries per (task, shot count)
MAX_NEW_TOKENS = 16
SEED = 0

DATA_DIR = Path("datasets")
OUT_DIR = Path("/mnt/user-data/outputs")
RESULTS_CACHE = OUT_DIR / "fv_shot_sweep_results.json"
CASE_SENSITIVE_TASKS = {"capitalize"}


# %%
# --------------------------------------------------------------------------
# Per-task dataset loaders -- three different source formats, so three loaders
# --------------------------------------------------------------------------

def load_antonym(path: Path) -> list[dict]:
    """AntSynNET (Nguyen et al. 2017). The repo's raw files are tab-separated
    word pairs, sometimes with a third column marking the relation type
    ('ant'/'syn' or similar). This assumes that layout -- open the actual
    downloaded file in datasets/ and confirm the column order/count before
    trusting this loader.
    """
    pairs = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 2:
                continue
            if len(parts) >= 3 and parts[2].lower() not in ("ant", "antonym", "antonyms"):
                continue  # skip synonym rows etc. if the file mixes relation types
            pairs.append({"input": parts[0], "output": parts[1]})
    return pairs


def load_english_french(paths) -> list[dict]:
    """Conneau et al. 2017 (MUSE) bilingual dictionary format: one
    'english_word french_word' pair per line, whitespace-separated.
    Accepts a single path or a list of paths -- MUSE ships en-fr as
    separate train (0-5000) / test (5000-6500) splits, but nothing in
    this pipeline trains/evaluates a mapping, so both are concatenated
    into one pool rather than kept separate.

    MUSE dictionaries list multiple valid translations for some English
    words (e.g. 'chat' -> 'discuter'/'discussion'/'chat'/...). Only the
    FIRST translation seen for a given English word is kept -- later
    duplicates are dropped -- so every English word maps to exactly one
    canonical French output. This avoids two problems: (a) the same
    input appearing as both a demonstration and the query with
    conflicting gold answers, and (b) is_correct() marking a model's
    valid-but-different translation as wrong non-deterministically.
    It does NOT fix the underlying scoring looseness -- a model that
    produces a *dropped* valid translation still scores as incorrect.
    If that turns out to matter, switch to scoring against the full
    set of valid translations per word instead of exact-match-one-string.
    """
    if isinstance(paths, (str, Path)):
        paths = [paths]
    seen = {}
    for path in paths:
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                en, fr = parts[0], parts[1]
                seen.setdefault(en, fr)
    return [{"input": en, "output": fr} for en, fr in seen.items()]


def load_json_pairs(path: Path) -> list[dict]:
    """ChatGPT-generated tasks (capitalize, country-capital, present-past,
    singular-plural). Expects a JSON list of {"input": ..., "output": ...}.
    """
    with open(path) as f:
        raw = json.load(f)
    if isinstance(raw, dict) and "examples" in raw:
        raw = raw["examples"]
    return [{"input": ex["input"], "output": ex["output"]} for ex in raw]


# task -> (loader function, path(s) relative to DATA_DIR)
# adjust filenames once you know what your antonym download is actually named
TASK_LOADERS = {
    "antonym": (load_antonym, "antonym_pairs.tsv"),
    "capitalize": (load_json_pairs, "capitalize.json"),
    "country-capital": (load_json_pairs, "country-capital.json"),
    "english-french": (load_english_french, ["en-fr.0-5000.txt", "en-fr.5000-6500.txt"]),
    "present-past": (load_json_pairs, "present-past.json"),
    "singular-plural": (load_json_pairs, "singular-plural.json"),
}


def load_fv_task(task_name: str, data_dir: Path = DATA_DIR) -> list[dict]:
    loader, rel_path = TASK_LOADERS[task_name]
    if isinstance(rel_path, list):
        full_paths = [data_dir / p for p in rel_path]
    else:
        full_paths = data_dir / rel_path
    return loader(full_paths)


# %%
# --------------------------------------------------------------------------
# Prompting / scoring
# --------------------------------------------------------------------------

def build_icl_prompt(pairs: list[dict], query_idx: int, n_shots: int, rng: random.Random) -> tuple[str, str]:
    """Excludes candidate demonstrations by matching INPUT WORD, not just
    index -- some datasets (e.g. english-french before dedup, or any task
    with accidental duplicate inputs) can otherwise let the same word
    appear as both a demonstration and the query, or as two demonstrations
    with different gold outputs, which confuses the in-context signal.
    """
    query_word = pairs[query_idx]["input"]
    pool = [i for i in range(len(pairs)) if i != query_idx and pairs[i]["input"] != query_word]
    shot_idxs = rng.sample(pool, min(n_shots, len(pool)))
    lines = [f"Q: {pairs[i]['input']}\nA: {pairs[i]['output']}" for i in shot_idxs]
    lines.append(f"Q: {query_word}\nA:")
    prompt = "\n\n".join(lines)
    gold = pairs[query_idx]["output"]
    return prompt, gold


def extract_word_answer(text: str) -> str:
    first_line = text.strip().split("\n")[0]
    return first_line.strip(string.punctuation + " ")


def is_correct(pred: str, gold: str, case_sensitive: bool) -> bool:
    if not case_sensitive:
        pred, gold = pred.lower(), gold.lower()
    return pred == gold


# %%
# --------------------------------------------------------------------------
# One (task, n_shots) configuration
# --------------------------------------------------------------------------

def run_config(task_name: str, n_shots: int, pairs: list[dict], rng: random.Random) -> dict:
    n = min(N_PER_CONFIG, len(pairs))
    query_idxs = rng.sample(range(len(pairs)), n)
    case_sensitive = task_name in CASE_SENSITIVE_TASKS

    nla_correct = 0
    para_correct = 0

    for qi in query_idxs:
        prompt, gold = build_icl_prompt(pairs, qi, n_shots, rng)

        # baseline run also captures h_orig -- required even though baseline
        # accuracy itself isn't scored/plotted here
        _, h_orig = run_with_patch(prompt, patch_vec=None, max_new_tokens=MAX_NEW_TOKENS)

        explanation = av(h_orig)

        h_hat = ar(explanation)
        text_nla, _ = run_with_patch(prompt, patch_vec=h_hat, max_new_tokens=MAX_NEW_TOKENS)
        nla_correct += is_correct(extract_word_answer(text_nla), gold, case_sensitive)

        paraphrased = paraphrase(explanation)
        h_hat_para = ar(paraphrased)
        text_para, _ = run_with_patch(prompt, patch_vec=h_hat_para, max_new_tokens=MAX_NEW_TOKENS)
        para_correct += is_correct(extract_word_answer(text_para), gold, case_sensitive)

    return {
        "n_shots": n_shots,
        "n": n,
        "acc_nla": nla_correct / n,
        "acc_para": para_correct / n,
    }


# %%
# --------------------------------------------------------------------------
# Sweep + plotting
# --------------------------------------------------------------------------

def run_sweep() -> dict:
    rng = random.Random(SEED)
    results = {}
    for task_name in TASKS:
        pairs = load_fv_task(task_name)
        results[task_name] = []
        for k in tqdm(SHOT_COUNTS, desc=task_name, unit="shot-count"):
            results[task_name].append(run_config(task_name, k, pairs, rng))
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_CACHE, "w") as f:
        json.dump(results, f, indent=2)
    return results


def plot_sweep(results: dict):
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    for task_name, rows in results.items():
        shots = [r["n_shots"] for r in rows]
        acc_nla = [r["acc_nla"] for r in rows]
        acc_para = [r["acc_para"] for r in rows]

        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(shots, acc_nla, marker="o", label="NLA round-trip")
        ax.plot(shots, acc_para, marker="s", label="Paraphrase round-trip")
        ax.set_xlabel("Number of shots")
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0, 1)
        ax.set_title(task_name)
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"fv_shot_sweep_{task_name}.png", dpi=150)
        plt.show()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
    for ax, task_name in zip(axes.flat, TASKS):
        rows = results[task_name]
        shots = [r["n_shots"] for r in rows]
        ax.plot(shots, [r["acc_nla"] for r in rows], marker="o", label="NLA round-trip")
        ax.plot(shots, [r["acc_para"] for r in rows], marker="s", label="Paraphrase round-trip")
        ax.set_title(task_name)
        ax.set_xlabel("Number of shots")
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
    axes.flat[0].legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / "fv_shot_sweep_all_tasks.png", dpi=150)
    plt.show()


# %%
# Run it. Comment out run_sweep() and load from RESULTS_CACHE instead
# if you're just re-plotting an already-completed sweep.
results = run_sweep()
plot_sweep(results)

## Cleanup

Close the AV client's HTTP session and stop SGLang only if this notebook launched it. An AV server that was already running before the notebook is left untouched.


In [8]:
if "client" in globals() and hasattr(client, "_http"):
    client._http.close()
    print("Closed NLA client HTTP session")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if "sglang_proc" in globals() and sglang_proc is not None:
    sglang_proc.terminate()
    sglang_proc.wait(timeout=30)
    print("Stopped SGLang launched by this notebook")
else:
    print("No SGLang subprocess launched by this notebook")

# Optional final memory check:
# !nvidia-smi

Closed NLA client HTTP session
Stopped SGLang launched by this notebook
